In [4]:
# =================================================================================================
# NHANES 2017-2018 LAB MERGE - FIDELITY PRESERVING
#
# Source corrected CSV files:
#
#   GLU_J_corrected.csv
#   INS_J_corrected.csv
#   TCHOL_J_corrected.csv
#   FASTQX_J_corrected.csv
#   GHB_J_corrected.csv
#   HDL_J_corrected.csv
#   TRIGLY_J_corrected.csv
#
#
# IMPORTANT 2017-2018 DESIGN
# -------------------------------------------------------------------------------------------------
#
# GLU_J DOES NOT contain:
#
#   PHAFSTHR
#   PHAFSTMN
#
#
# INS_J ALSO DOES NOT contain:
#
#   PHAFSTHR
#   PHAFSTMN
#
#
# CDC directs users to FASTQX_J for these variables.
#
#
# Therefore:
#
#   PHAFSTHR
#   PHAFSTMN
#
# are retained directly from FASTQX_J.
#
# There is NO:
#
#   GLU_J <-> FASTQX_J PHAFSTHR comparison
#
# and NO:
#
#   INS_J <-> FASTQX_J PHAFSTHR comparison
#
#
# WTSAF2YR
# -------------------------------------------------------------------------------------------------
#
# Exists in:
#
#   GLU_J
#   INS_J
#   TRIGLY_J
#
# All three contain the same fasting-subsample participant universe.
#
# Strategy:
#
#   1. Validate same SEQN set.
#   2. Validate missing positions.
#   3. Validate NUMERIC equality.
#   4. Keep GLU_J as canonical WTSAF2YR.
#   5. Drop WTSAF2YR from INS_J before merge.
#   6. Drop WTSAF2YR from TRIGLY_J before merge.
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
# - Read corrected CSVs with dtype=str
# - Preserve decimal text AS IS
# - Never round
# - Never use float_format=
# - Never integerize genuine Lab measurements
# - SEQN remains integer-looking text
# - Every source has unique SEQN
# - Every merge uses validate="one_to_one"
# - No uncontrolled pandas suffixes
# - Exact retained source text checked after merge
# - Exact pre-save/post-save text checked
# - Known tiny XPORT token must be absent
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


LAB_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Labs"
)


GLU_FILE = (
    LAB_DIR
    / "GLU_J_corrected.csv"
)


INS_FILE = (
    LAB_DIR
    / "INS_J_corrected.csv"
)


TCHOL_FILE = (
    LAB_DIR
    / "TCHOL_J_corrected.csv"
)


FASTQX_FILE = (
    LAB_DIR
    / "FASTQX_J_corrected.csv"
)


GHB_FILE = (
    LAB_DIR
    / "GHB_J_corrected.csv"
)


HDL_FILE = (
    LAB_DIR
    / "HDL_J_corrected.csv"
)


TRIGLY_FILE = (
    LAB_DIR
    / "TRIGLY_J_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2017_2018"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_labs_2017_2018_raw_corrected.csv"
)


# =================================================================================================
# 3. VERIFY SOURCE FILES EXIST
# =================================================================================================

FILES = {

    "GLU_J": GLU_FILE,

    "INS_J": INS_FILE,

    "TCHOL_J": TCHOL_FILE,

    "FASTQX_J": FASTQX_FILE,

    "GHB_J": GHB_FILE,

    "HDL_J": HDL_FILE,

    "TRIGLY_J": TRIGLY_FILE

}


for name, path in FILES.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} corrected file not found:\n{path}"
        )


print(
    "PASS: All seven 2017-2018 Lab source files exist."
)


# =================================================================================================
# 4. READ CORRECTED CSV FILES AS TEXT
# =================================================================================================

#
# CRITICAL:
#
# dtype=str + keep_default_na=False
#
# prevents pandas from reformatting corrected decimal tokens.
#

glucose = pd.read_csv(
    GLU_FILE,
    dtype=str,
    keep_default_na=False
)


insulin = pd.read_csv(
    INS_FILE,
    dtype=str,
    keep_default_na=False
)


chol = pd.read_csv(
    TCHOL_FILE,
    dtype=str,
    keep_default_na=False
)


fasting = pd.read_csv(
    FASTQX_FILE,
    dtype=str,
    keep_default_na=False
)


ghb = pd.read_csv(
    GHB_FILE,
    dtype=str,
    keep_default_na=False
)


hdl = pd.read_csv(
    HDL_FILE,
    dtype=str,
    keep_default_na=False
)


trigly = pd.read_csv(
    TRIGLY_FILE,
    dtype=str,
    keep_default_na=False
)


# =================================================================================================
# 5. DATASET DICTIONARY
# =================================================================================================

datasets = {

    "GLU_J": glucose,

    "INS_J": insulin,

    "TCHOL_J": chol,

    "FASTQX_J": fasting,

    "GHB_J": ghb,

    "HDL_J": hdl,

    "TRIGLY_J": trigly

}


# =================================================================================================
# 6. STANDARDIZE COLUMN NAMES ONLY
# =================================================================================================

for name, dataset in datasets.items():

    dataset.columns = (
        dataset.columns
        .str.strip()
        .str.upper()
    )


# =================================================================================================
# 7. DISPLAY SOURCE STRUCTURE
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2017-2018 LAB MERGE"
)


print(
    "=" * 110
)


summary = []


for name, dataset in datasets.items():

    summary.append({

        "file": name,

        "rows": len(dataset),

        "columns": len(dataset.columns),

        "column_names":
            dataset.columns.tolist()

    })


summary_df = pd.DataFrame(
    summary
)


display(
    summary_df
)


# =================================================================================================
# 8. OFFICIAL CDC SOURCE ROW COUNTS
# =================================================================================================

EXPECTED_SOURCE_ROWS = {

    "GLU_J": 3036,

    "INS_J": 3036,

    "TRIGLY_J": 3036,

    "TCHOL_J": 7435,

    "HDL_J": 7435,

    "GHB_J": 6401,

    "FASTQX_J": 8366

}


print(
    "\n--- OFFICIAL SOURCE ROW COUNTS ---"
)


for name, expected_rows in EXPECTED_SOURCE_ROWS.items():

    actual_rows = len(
        datasets[name]
    )


    print(
        f"{name}: "
        f"expected={expected_rows:,}, "
        f"actual={actual_rows:,}"
    )


    if actual_rows != expected_rows:

        raise ValueError(
            f"{name} row count differs from CDC.\n"
            f"Expected: {expected_rows:,}\n"
            f"Actual:   {actual_rows:,}"
        )


print(
    "PASS: All source row counts match "
    "2017-2018 expectations."
)


# =================================================================================================
# 9. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataset,
    dataset_name
):

    print(
        f"\n--- SEQN CHECK: {dataset_name} ---"
    )


    if "SEQN" not in dataset.columns:

        raise ValueError(
            f"{dataset_name}: SEQN missing."
        )


    blank_count = int(

        dataset["SEQN"]
        .eq("")
        .sum()

    )


    duplicate_count = int(

        dataset["SEQN"]
        .duplicated()
        .sum()

    )


    dot_zero_count = int(

        (
            dataset["SEQN"].ne("")
            &
            dataset["SEQN"]
            .str.endswith(".0")
        )

        .sum()

    )


    invalid_format_count = int(

        (
            dataset["SEQN"].ne("")
            &
            ~dataset["SEQN"]
            .str.fullmatch(
                r"\d+"
            )
        )

        .sum()

    )


    print(
        "Rows:",
        f"{len(dataset):,}"
    )


    print(
        "Unique SEQN:",
        f"{dataset['SEQN'].nunique():,}"
    )


    print(
        "Blank SEQN:",
        blank_count
    )


    print(
        "Duplicate SEQN:",
        duplicate_count
    )


    print(
        "SEQN ending '.0':",
        dot_zero_count
    )


    print(
        "Invalid format:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{dataset_name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{dataset_name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{dataset_name}: SEQN contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{dataset_name}: invalid SEQN formatting."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 10. VALIDATE SEQN IN ALL SOURCES
# =================================================================================================

for name, dataset in datasets.items():

    validate_seqn(
        dataset,
        name
    )


print(
    "\nPASS: SEQN integrity validated "
    "in all seven Lab files."
)


# =================================================================================================
# 11. VERIFY IMPORTANT 2017-2018 SOURCE DESIGN
# =================================================================================================

#
# CRITICAL FIX FOR YOUR ERROR:
#
# Neither GLU_J nor INS_J should contain:
#
#   PHAFSTHR
#   PHAFSTMN
#

for source_name, source_df in {

    "GLU_J": glucose,

    "INS_J": insulin

}.items():

    for col in [

        "PHAFSTHR",

        "PHAFSTMN"

    ]:

        if col in source_df.columns:

            raise ValueError(
                f"Unexpected {col} found in {source_name}.\n"
                "For 2017-2018 these fasting-duration "
                "variables belong to FASTQX_J."
            )


print(
    "\nPASS: GLU_J and INS_J correctly do NOT contain "
    "PHAFSTHR / PHAFSTMN."
)


#
# FASTQX_J MUST contain them.
#

for col in [

    "PHAFSTHR",

    "PHAFSTMN"

]:

    if col not in fasting.columns:

        raise ValueError(
            f"{col} missing from FASTQX_J."
        )


print(
    "PASS: FASTQX_J contains canonical "
    "PHAFSTHR / PHAFSTMN."
)


# =================================================================================================
# 12. VERIFY REQUIRED SOURCE VARIABLES
# =================================================================================================

# -------------------------------------------------------------------------------------------------
# GLU_J
# -------------------------------------------------------------------------------------------------

for col in [

    "SEQN",

    "WTSAF2YR",

    "LBXGLU",

    "LBDGLUSI"

]:

    if col not in glucose.columns:

        raise ValueError(
            f"Expected GLU_J variable missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# INS_J
# -------------------------------------------------------------------------------------------------

for col in [

    "SEQN",

    "WTSAF2YR",

    "LBXIN",

    "LBDINSI",

    "LBDINLC"

]:

    if col not in insulin.columns:

        raise ValueError(
            f"Expected INS_J variable missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# TRIGLY_J
# -------------------------------------------------------------------------------------------------

for col in [

    "SEQN",

    "WTSAF2YR",

    "LBXTR",

    "LBDTRSI",

    "LBDLDL",

    "LBDLDLSI",

    "LBDLDLM",

    "LBDLDMSI",

    "LBDLDLN",

    "LBDLDNSI"

]:

    if col not in trigly.columns:

        raise ValueError(
            f"Expected TRIGLY_J variable missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# GHB_J
# -------------------------------------------------------------------------------------------------

if "LBXGH" not in ghb.columns:

    raise ValueError(
        "LBXGH missing from GHB_J."
    )


# -------------------------------------------------------------------------------------------------
# HDL_J
# -------------------------------------------------------------------------------------------------

for col in [

    "LBDHDD",

    "LBDHDDSI"

]:

    if col not in hdl.columns:

        raise ValueError(
            f"Expected HDL_J variable missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# TCHOL_J
# -------------------------------------------------------------------------------------------------

for col in [

    "LBXTC",

    "LBDTCSI"

]:

    if col not in chol.columns:

        raise ValueError(
            f"Expected TCHOL_J variable missing: {col}"
        )


print(
    "PASS: Key source structures validated."
)


# =================================================================================================
# 13. HELPER - NONBLANK COUNT
# =================================================================================================

def count_nonblank(
    dataset,
    column
):

    if column not in dataset.columns:

        raise ValueError(
            f"Missing field: {column}"
        )


    return int(

        dataset[column]
        .ne("")
        .sum()

    )


# =================================================================================================
# 14. CDC KEY RESULT COUNTS
# =================================================================================================

#
# GLU_J
#

if count_nonblank(
    glucose,
    "LBXGLU"
) != 2891:

    raise ValueError(
        "GLU_J LBXGLU count differs from CDC."
    )


if count_nonblank(
    glucose,
    "LBDGLUSI"
) != 2891:

    raise ValueError(
        "GLU_J LBDGLUSI count differs from CDC."
    )


#
# INS_J
#

if count_nonblank(
    insulin,
    "LBXIN"
) != 2825:

    raise ValueError(
        "INS_J LBXIN count differs from CDC."
    )


if count_nonblank(
    insulin,
    "LBDINSI"
) != 2825:

    raise ValueError(
        "INS_J LBDINSI count differs from CDC."
    )


if count_nonblank(
    insulin,
    "LBDINLC"
) != 2825:

    raise ValueError(
        "INS_J LBDINLC count differs from CDC."
    )


#
# TRIGLY_J
#

if count_nonblank(
    trigly,
    "LBXTR"
) != 2834:

    raise ValueError(
        "TRIGLY_J LBXTR count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDTRSI"
) != 2834:

    raise ValueError(
        "TRIGLY_J LBDTRSI count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDL"
) != 2808:

    raise ValueError(
        "TRIGLY_J LBDLDL count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDLSI"
) != 2808:

    raise ValueError(
        "TRIGLY_J LBDLDLSI count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDLM"
) != 2808:

    raise ValueError(
        "TRIGLY_J LBDLDLM count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDMSI"
) != 2808:

    raise ValueError(
        "TRIGLY_J LBDLDMSI count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDLN"
) != 2827:

    raise ValueError(
        "TRIGLY_J LBDLDLN count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDNSI"
) != 2827:

    raise ValueError(
        "TRIGLY_J LBDLDNSI count differs from CDC."
    )


#
# FASTQX_J
#

if count_nonblank(
    fasting,
    "PHAFSTHR"
) != 7996:

    raise ValueError(
        "FASTQX_J PHAFSTHR count differs from CDC."
    )


if count_nonblank(
    fasting,
    "PHAFSTMN"
) != 7996:

    raise ValueError(
        "FASTQX_J PHAFSTMN count differs from CDC."
    )


#
# GHB_J
#

if count_nonblank(
    ghb,
    "LBXGH"
) != 6045:

    raise ValueError(
        "GHB_J LBXGH count differs from CDC."
    )


#
# TCHOL_J
#

if count_nonblank(
    chol,
    "LBXTC"
) != 6738:

    raise ValueError(
        "TCHOL_J LBXTC count differs from CDC."
    )


if count_nonblank(
    chol,
    "LBDTCSI"
) != 6738:

    raise ValueError(
        "TCHOL_J LBDTCSI count differs from CDC."
    )


#
# HDL_J
#

if count_nonblank(
    hdl,
    "LBDHDD"
) != 6738:

    raise ValueError(
        "HDL_J LBDHDD count differs from CDC."
    )


if count_nonblank(
    hdl,
    "LBDHDDSI"
) != 6738:

    raise ValueError(
        "HDL_J LBDHDDSI count differs from CDC."
    )


print(
    "\nPASS: Key 2017-2018 Lab result counts match CDC."
)


# =================================================================================================
# 15. WTSAF2YR CDC DISTRIBUTION CHECK
# =================================================================================================

#
# Official 2017-2018 fasting subsample:
#
# rows                 = 3,036
# positive WTSAF2YR    = 2,711
# WTSAF2YR = 0         =   325
# missing              =     0
#

def validate_wtsaf2yr(
    dataset,
    dataset_name
):

    if "WTSAF2YR" not in dataset.columns:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR missing."
        )


    values = pd.to_numeric(

        dataset["WTSAF2YR"]
        .where(
            dataset["WTSAF2YR"] != "",
            np.nan
        ),

        errors="raise"

    )


    positive_count = int(

        (
            values > 0
        )

        .sum()

    )


    zero_count = int(

        values
        .eq(0)
        .sum()

    )


    missing_count = int(

        values
        .isna()
        .sum()

    )


    print(
        f"\n--- WTSAF2YR CHECK: {dataset_name} ---"
    )


    print(
        "Positive:",
        f"{positive_count:,}"
    )


    print(
        "Zero:",
        f"{zero_count:,}"
    )


    print(
        "Missing:",
        f"{missing_count:,}"
    )


    if positive_count != 2711:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR positive count "
            "differs from CDC."
        )


    if zero_count != 325:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR zero count "
            "differs from CDC."
        )


    if missing_count != 0:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR missing count "
            "differs from CDC."
        )


    print(
        "PASS"
    )


validate_wtsaf2yr(
    glucose,
    "GLU_J"
)


validate_wtsaf2yr(
    insulin,
    "INS_J"
)


validate_wtsaf2yr(
    trigly,
    "TRIGLY_J"
)


print(
    "\nPASS: WTSAF2YR distribution validated "
    "in GLU_J, INS_J and TRIGLY_J."
)


# =================================================================================================
# 16. DISPLAY CROSS-FILE VARIABLE OVERLAPS
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "CROSS-FILE VARIABLE OVERLAPS"
)


print(
    "=" * 110
)


dataset_names = list(
    datasets.keys()
)


for i in range(
    len(dataset_names)
):

    for j in range(
        i + 1,
        len(dataset_names)
    ):

        left_name = (
            dataset_names[i]
        )


        right_name = (
            dataset_names[j]
        )


        overlap = sorted(

            (
                set(
                    datasets[
                        left_name
                    ].columns
                )

                &

                set(
                    datasets[
                        right_name
                    ].columns
                )
            )

            -

            {
                "SEQN"
            }

        )


        if overlap:

            print(
                f"{left_name} <-> {right_name}:",
                overlap
            )


# =================================================================================================
# 17. NUMERIC OVERLAP COMPARISON FUNCTION
# =================================================================================================

#
# Corrected CSVs remain strings.
#
# Temporary numeric copies are used ONLY for equality validation.
#
# Therefore:
#
#   "8"
#
# and:
#
#   "8.0"
#
# are treated as scientifically equivalent.
#

def compare_numeric_variable_by_seqn(
    left_df,
    right_df,
    column,
    left_name,
    right_name,
    require_same_seqn_set=False,
    require_common_missing_match=False
):

    print(
        "\n"
        + "-" * 105
    )


    print(
        f"NUMERIC FIDELITY CHECK: "
        f"{left_name}.{column} "
        f"<-> "
        f"{right_name}.{column}"
    )


    print(
        "-" * 105
    )


    if column not in left_df.columns:

        raise ValueError(
            f"{column} missing from {left_name}."
        )


    if column not in right_df.columns:

        raise ValueError(
            f"{column} missing from {right_name}."
        )


    left_part = (

        left_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}_LEFT"
            }
        )

    )


    right_part = (

        right_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}_RIGHT"
            }
        )

    )


    comparison = (

        left_part

        .merge(

            right_part,

            on="SEQN",

            how="outer",

            validate="one_to_one",

            indicator=True

        )

    )


    comparison[
        f"{column}_LEFT"
    ] = (

        comparison[
            f"{column}_LEFT"
        ]

        .fillna("")

    )


    comparison[
        f"{column}_RIGHT"
    ] = (

        comparison[
            f"{column}_RIGHT"
        ]

        .fillna("")

    )


    only_left = int(

        comparison["_merge"]
        .eq("left_only")
        .sum()

    )


    only_right = int(

        comparison["_merge"]
        .eq("right_only")
        .sum()

    )


    common = (
        comparison["_merge"]
        .eq("both")
    )


    common_count = int(
        common.sum()
    )


    print(
        "Common SEQN:",
        f"{common_count:,}"
    )


    print(
        "Only in",
        left_name,
        ":",
        f"{only_left:,}"
    )


    print(
        "Only in",
        right_name,
        ":",
        f"{only_right:,}"
    )


    if require_same_seqn_set:

        if (
            only_left != 0
            or
            only_right != 0
        ):

            raise ValueError(
                f"{left_name} and {right_name} "
                f"do not contain the same SEQN set "
                f"for {column}."
            )


    common_left_raw = (

        comparison.loc[
            common,
            f"{column}_LEFT"
        ]

        .reset_index(
            drop=True
        )

    )


    common_right_raw = (

        comparison.loc[
            common,
            f"{column}_RIGHT"
        ]

        .reset_index(
            drop=True
        )

    )


    left_populated = (
        common_left_raw.ne("")
    )


    right_populated = (
        common_right_raw.ne("")
    )


    common_missing_mismatch = int(

        (
            left_populated
            !=
            right_populated
        )

        .sum()

    )


    print(
        "Missing-position mismatches:",
        f"{common_missing_mismatch:,}"
    )


    if (
        require_common_missing_match
        and
        common_missing_mismatch != 0
    ):

        raise ValueError(
            f"{column}: missing positions differ between "
            f"{left_name} and {right_name}."
        )


    both_populated = (
        left_populated
        &
        right_populated
    )


    overlap_count = int(
        both_populated.sum()
    )


    print(
        "Both populated:",
        f"{overlap_count:,}"
    )


    left_numeric = pd.to_numeric(

        common_left_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    right_numeric = pd.to_numeric(

        common_right_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        left_numeric.to_numpy(
            dtype=float
        ),

        right_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(

        (
            ~numeric_equal
        )

        .sum()

    )


    left_text = (

        common_left_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    right_text = (

        common_right_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    text_difference_count = int(

        (
            left_text
            !=
            right_text
        )

        .sum()

    )


    print(
        "Raw text-format differences:",
        f"{text_difference_count:,}"
    )


    print(
        "TRUE numeric mismatches:",
        f"{numeric_mismatch_count:,}"
    )


    if numeric_mismatch_count != 0:

        common_rows = (

            comparison.loc[
                common
            ]

            .reset_index(
                drop=True
            )

        )


        populated_rows = (

            common_rows.loc[
                both_populated.to_numpy()
            ]

            .reset_index(
                drop=True
            )

        )


        populated_rows[
            f"{column}_LEFT_NUMERIC"
        ] = (

            left_numeric
            .reset_index(
                drop=True
            )

        )


        populated_rows[
            f"{column}_RIGHT_NUMERIC"
        ] = (

            right_numeric
            .reset_index(
                drop=True
            )

        )


        populated_rows[
            "NUMERIC_MATCH"
        ] = numeric_equal


        print(
            "\nTRUE NUMERIC MISMATCH EXAMPLES:"
        )


        display(

            populated_rows.loc[
                ~populated_rows[
                    "NUMERIC_MATCH"
                ],
                [
                    "SEQN",

                    f"{column}_LEFT",

                    f"{column}_RIGHT",

                    f"{column}_LEFT_NUMERIC",

                    f"{column}_RIGHT_NUMERIC"
                ]
            ]

            .head(20)

        )


        raise ValueError(
            f"{column} has true numeric differences "
            f"between {left_name} and {right_name}."
        )


    print(
        "PASS: Numeric values agree."
    )


    return {

        "common_seqn":
            common_count,

        "overlap":
            overlap_count,

        "common_missing_mismatches":
            common_missing_mismatch,

        "text_differences":
            text_difference_count,

        "numeric_mismatches":
            numeric_mismatch_count,

        "only_left_seqn":
            only_left,

        "only_right_seqn":
            only_right

    }


# =================================================================================================
# 18. VALIDATE WTSAF2YR - GLU_J VS INS_J
# =================================================================================================

wtsaf_ins_result = (

    compare_numeric_variable_by_seqn(

        glucose,

        insulin,

        "WTSAF2YR",

        "GLU_J",

        "INS_J",

        require_same_seqn_set=True,

        require_common_missing_match=True

    )

)


# =================================================================================================
# 19. VALIDATE WTSAF2YR - GLU_J VS TRIGLY_J
# =================================================================================================

wtsaf_trigly_result = (

    compare_numeric_variable_by_seqn(

        glucose,

        trigly,

        "WTSAF2YR",

        "GLU_J",

        "TRIGLY_J",

        require_same_seqn_set=True,

        require_common_missing_match=True

    )

)


print(
    "\nPASS: WTSAF2YR is numerically equivalent "
    "across GLU_J, INS_J and TRIGLY_J."
)


print(
    "Canonical WTSAF2YR source: GLU_J"
)


# =================================================================================================
# 20. NO PHAFSTHR / PHAFSTMN CROSS-FILE COMPARISON
# =================================================================================================

#
# THIS IS THE FIX FOR THE ERROR YOU REPORTED.
#
# DO NOT run:
#
#   compare_numeric_variable_by_seqn(
#       glucose,
#       fasting,
#       "PHAFSTHR",
#       ...
#   )
#
# GLU_J does not contain PHAFSTHR.
#
# DO NOT compare INS_J either.
#
# INS_J does not contain PHAFSTHR.
#
# FASTQX_J is the canonical source.
#

print(
    "\n2017-2018 fasting-duration design:"
)


print(
    "  GLU_J:    PHAFSTHR / PHAFSTMN absent"
)


print(
    "  INS_J:    PHAFSTHR / PHAFSTMN absent"
)


print(
    "  FASTQX_J: canonical PHAFSTHR / PHAFSTMN"
)


print(
    "PASS: No inappropriate PHAFSTHR cross-file comparison."
)


# =================================================================================================
# 21. PREPARE INS_J FOR MERGE
# =================================================================================================

#
# WTSAF2YR has already been validated against GLU_J.
#

insulin_for_merge = (

    insulin.drop(
        columns=[
            "WTSAF2YR"
        ]
    )

)


print(
    "\nRemoved validated redundant INS_J field:"
)


print(
    " - WTSAF2YR"
)


# =================================================================================================
# 22. PREPARE TRIGLY_J FOR MERGE
# =================================================================================================

trigly_for_merge = (

    trigly.drop(
        columns=[
            "WTSAF2YR"
        ]
    )

)


print(
    "\nRemoved validated redundant TRIGLY_J field:"
)


print(
    " - WTSAF2YR"
)


# =================================================================================================
# 23. FASTQX_J REMAINS CANONICAL
# =================================================================================================

#
# DO NOT rename:
#
#   PHAFSTHR -> PHAFSTHR_FAST
#   PHAFSTMN -> PHAFSTMN_FAST
#
# There is no competing GLU_J or INS_J field.
#

fasting_for_merge = (
    fasting.copy()
)


print(
    "\nFASTQX_J canonical fasting fields:"
)


print(
    " - PHAFSTHR"
)


print(
    " - PHAFSTMN"
)


# =================================================================================================
# 24. SAFE OUTER MERGE
# =================================================================================================

def safe_outer_merge(
    base,
    incoming,
    incoming_name
):

    overlap = sorted(

        (
            set(
                base.columns
            )

            &

            set(
                incoming.columns
            )
        )

        -

        {
            "SEQN"
        }

    )


    if overlap:

        raise ValueError(
            f"Unexpected overlapping columns before "
            f"merging {incoming_name}:\n"
            f"{overlap}"
        )


    result = (

        base.merge(

            incoming,

            on="SEQN",

            how="outer",

            validate="one_to_one"

        )

    )


    return result


# =================================================================================================
# 25. EXPECTED FINAL SEQN UNION
# =================================================================================================

expected_seqn_union = set()


for dataset in datasets.values():

    expected_seqn_union.update(
        dataset["SEQN"]
    )


print(
    "\nExpected final unique Lab SEQN:",
    f"{len(expected_seqn_union):,}"
)


#
# FASTQX_J is the broadest source.
#

if len(expected_seqn_union) != 8366:

    raise ValueError(
        "Unexpected source SEQN union.\n"
        f"Expected: 8,366\n"
        f"Actual:   {len(expected_seqn_union):,}"
    )


print(
    "PASS: Source SEQN union = 8,366."
)


# =================================================================================================
# 26. START WITH GLU_J
# =================================================================================================

df = (
    glucose.copy()
)


# =================================================================================================
# 27. MERGE INS_J
# =================================================================================================

df = safe_outer_merge(

    df,

    insulin_for_merge,

    "INS_J"

)


print(
    "Merged INS_J"
)


# =================================================================================================
# 28. MERGE TCHOL_J
# =================================================================================================

df = safe_outer_merge(

    df,

    chol,

    "TCHOL_J"

)


print(
    "Merged TCHOL_J"
)


# =================================================================================================
# 29. MERGE FASTQX_J
# =================================================================================================

df = safe_outer_merge(

    df,

    fasting_for_merge,

    "FASTQX_J"

)


print(
    "Merged FASTQX_J"
)


# =================================================================================================
# 30. MERGE GHB_J
# =================================================================================================

df = safe_outer_merge(

    df,

    ghb,

    "GHB_J"

)


print(
    "Merged GHB_J"
)


# =================================================================================================
# 31. MERGE HDL_J
# =================================================================================================

df = safe_outer_merge(

    df,

    hdl,

    "HDL_J"

)


print(
    "Merged HDL_J"
)


# =================================================================================================
# 32. MERGE TRIGLY_J
# =================================================================================================

df = safe_outer_merge(

    df,

    trigly_for_merge,

    "TRIGLY_J"

)


print(
    "Merged TRIGLY_J"
)


# =================================================================================================
# 33. NORMALIZE OUTER-MERGE MISSING VALUES
# =================================================================================================

df = (
    df.fillna("")
)


# =================================================================================================
# 34. FINAL SEQN VALIDATION
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "FINAL SEQN CHECK"
)


print(
    "=" * 110
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Unique SEQN:",
    f"{df['SEQN'].nunique():,}"
)


if len(df) != 8366:

    raise ValueError(
        "Final Lab row count differs.\n"
        f"Expected: 8,366\n"
        f"Actual:   {len(df):,}"
    )


if (
    df["SEQN"]
    .nunique()
    != 8366
):

    raise ValueError(
        "Final unique SEQN differs from expected."
    )


if (
    df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final Lab dataset contains duplicate SEQN."
    )


if (
    df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final Lab dataset contains blank SEQN."
    )


if (
    df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final Lab dataset contains SEQN '.0'."
    )


if (

    ~df["SEQN"]
    .str.fullmatch(
        r"\d+"
    )

).any():

    raise ValueError(
        "Final Lab dataset contains invalid SEQN."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 35. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_column_names = (

    df.columns[
        df.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_column_names
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 36. VERIFY ONE CANONICAL WTSAF2YR
# =================================================================================================

final_weight_columns = [

    col

    for col in df.columns

    if col.startswith(
        "WTSAF2YR"
    )

]


print(
    "\nWTSAF2YR fields:"
)


print(
    final_weight_columns
)


if final_weight_columns != [

    "WTSAF2YR"

]:

    raise ValueError(
        "Expected exactly one WTSAF2YR field."
    )


print(
    "PASS: Exactly one canonical WTSAF2YR retained "
    "from GLU_J."
)


# =================================================================================================
# 37. VERIFY CANONICAL FASTING FIELDS
# =================================================================================================

for col in [

    "PHAFSTHR",

    "PHAFSTMN"

]:

    if col not in df.columns:

        raise ValueError(
            f"Canonical FASTQX_J field missing: {col}"
        )


BAD_FASTING_FIELDS = [

    "PHAFSTHR_FAST",

    "PHAFSTMN_FAST",

    "PHAFSTHR_GLU",

    "PHAFSTMN_GLU",

    "PHAFSTHR_INS",

    "PHAFSTMN_INS",

    "PHAFSTHR_X",

    "PHAFSTHR_Y",

    "PHAFSTMN_X",

    "PHAFSTMN_Y"

]


bad_fasting_fields_found = [

    col

    for col in BAD_FASTING_FIELDS

    if col in df.columns

]


if bad_fasting_fields_found:

    raise ValueError(
        "Duplicate/engineered fasting fields remain:\n"
        + str(
            bad_fasting_fields_found
        )
    )


print(
    "PASS: One canonical PHAFSTHR / PHAFSTMN pair "
    "retained from FASTQX_J."
)


# =================================================================================================
# 38. NO MISLEADING MERGE SUFFIXES
# =================================================================================================

BAD_SUFFIXES = [

    "_LBXGH",

    "_TRIGLY",

    "_INS",

    "_GLU",

    "_CHOL",

    "_HDL",

    "_X",

    "_Y"

]


unexpected_suffix_columns = []


for col in df.columns:

    if any(

        col.endswith(
            suffix
        )

        for suffix in BAD_SUFFIXES

    ):

        unexpected_suffix_columns.append(
            col
        )


if unexpected_suffix_columns:

    raise ValueError(
        "Unexpected merge-generated suffix columns:\n"
        + str(
            unexpected_suffix_columns
        )
    )


print(
    "PASS: No misleading merge-generated suffixes."
)


# =================================================================================================
# 39. EXACT SOURCE-TEXT FIDELITY FUNCTION
# =================================================================================================

def validate_source_text_fidelity(
    source_df,
    final_df,
    source_name,
    retained_columns
):

    print(
        f"\n--- SOURCE FIDELITY: {source_name} ---"
    )


    if not retained_columns:

        print(
            "No retained fields to validate."
        )

        return


    source_check = (

        source_df[
            ["SEQN"] + retained_columns
        ]

        .copy()

    )


    final_check = (

        final_df[
            ["SEQN"] + retained_columns
        ]

        .copy()

    )


    final_check = (

        final_check.rename(

            columns={

                col:
                    f"{col}__FINAL"

                for col in retained_columns

            }

        )

    )


    check = (

        source_check

        .merge(

            final_check,

            on="SEQN",

            how="left",

            validate="one_to_one"

        )

        .fillna("")

    )


    failures = []


    for col in retained_columns:

        mismatch_count = int(

            (
                check[col]
                !=
                check[
                    f"{col}__FINAL"
                ]
            )

            .sum()

        )


        if mismatch_count > 0:

            failures.append(
                (
                    col,
                    mismatch_count
                )
            )


    if failures:

        raise ValueError(
            f"{source_name} source-text fidelity failures:\n"
            + str(
                failures
            )
        )


    print(
        "PASS:",
        len(retained_columns),
        "retained fields match text-for-text."
    )


# =================================================================================================
# 40. GLU_J SOURCE FIDELITY
# =================================================================================================

GLU_RETAINED = [

    col

    for col in glucose.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    glucose,

    df,

    "GLU_J",

    GLU_RETAINED

)


# =================================================================================================
# 41. INS_J SOURCE FIDELITY
# =================================================================================================

#
# WTSAF2YR intentionally deduplicated.
#

INS_RETAINED = [

    col

    for col in insulin.columns

    if col not in {

        "SEQN",

        "WTSAF2YR"

    }

]


validate_source_text_fidelity(

    insulin,

    df,

    "INS_J",

    INS_RETAINED

)


# =================================================================================================
# 42. TCHOL_J SOURCE FIDELITY
# =================================================================================================

TCHOL_RETAINED = [

    col

    for col in chol.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    chol,

    df,

    "TCHOL_J",

    TCHOL_RETAINED

)


# =================================================================================================
# 43. FASTQX_J SOURCE FIDELITY
# =================================================================================================

FASTQX_RETAINED = [

    col

    for col in fasting.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    fasting,

    df,

    "FASTQX_J",

    FASTQX_RETAINED

)


# =================================================================================================
# 44. GHB_J SOURCE FIDELITY
# =================================================================================================

GHB_RETAINED = [

    col

    for col in ghb.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    ghb,

    df,

    "GHB_J",

    GHB_RETAINED

)


# =================================================================================================
# 45. HDL_J SOURCE FIDELITY
# =================================================================================================

HDL_RETAINED = [

    col

    for col in hdl.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    hdl,

    df,

    "HDL_J",

    HDL_RETAINED

)


# =================================================================================================
# 46. TRIGLY_J SOURCE FIDELITY
# =================================================================================================

TRIGLY_RETAINED = [

    col

    for col in trigly.columns

    if col not in {

        "SEQN",

        "WTSAF2YR"

    }

]


validate_source_text_fidelity(

    trigly,

    df,

    "TRIGLY_J",

    TRIGLY_RETAINED

)


# =================================================================================================
# 47. FINAL WTSAF2YR CHECK
# =================================================================================================

final_wtsaf = pd.to_numeric(

    df["WTSAF2YR"]
    .where(
        df["WTSAF2YR"] != "",
        np.nan
    ),

    errors="raise"

)


final_wtsaf_nonblank = int(

    df["WTSAF2YR"]
    .ne("")
    .sum()

)


final_wtsaf_positive = int(

    (
        final_wtsaf > 0
    )

    .sum()

)


final_wtsaf_zero = int(

    final_wtsaf
    .eq(0)
    .sum()

)


print(
    "\n--- FINAL WTSAF2YR CHECK ---"
)


print(
    "Nonblank:",
    f"{final_wtsaf_nonblank:,}"
)


print(
    "Positive:",
    f"{final_wtsaf_positive:,}"
)


print(
    "Zero:",
    f"{final_wtsaf_zero:,}"
)


if final_wtsaf_nonblank != 3036:

    raise ValueError(
        "Final WTSAF2YR populated count differs."
    )


if final_wtsaf_positive != 2711:

    raise ValueError(
        "Final WTSAF2YR positive count differs."
    )


if final_wtsaf_zero != 325:

    raise ValueError(
        "Final WTSAF2YR zero count differs."
    )


print(
    "PASS: Final WTSAF2YR matches CDC."
)


# =================================================================================================
# 48. FINAL FASTING TIME CHECK
# =================================================================================================

final_phafsthr_count = int(

    df["PHAFSTHR"]
    .ne("")
    .sum()

)


final_phafstmn_count = int(

    df["PHAFSTMN"]
    .ne("")
    .sum()

)


print(
    "\n--- FINAL FASTING TIME CHECK ---"
)


print(
    "PHAFSTHR populated:",
    f"{final_phafsthr_count:,}"
)


print(
    "PHAFSTMN populated:",
    f"{final_phafstmn_count:,}"
)


if final_phafsthr_count != 7996:

    raise ValueError(
        "Final PHAFSTHR count differs from FASTQX_J."
    )


if final_phafstmn_count != 7996:

    raise ValueError(
        "Final PHAFSTMN count differs from FASTQX_J."
    )


print(
    "PASS: FASTQX_J fasting-time coverage preserved."
)


# =================================================================================================
# 49. FINAL INSULIN CHECK
# =================================================================================================

if int(

    df["LBXIN"]
    .ne("")
    .sum()

) != 2825:

    raise ValueError(
        "Final LBXIN count differs from INS_J."
    )


if int(

    df["LBDINSI"]
    .ne("")
    .sum()

) != 2825:

    raise ValueError(
        "Final LBDINSI count differs from INS_J."
    )


if int(

    df["LBDINLC"]
    .ne("")
    .sum()

) != 2825:

    raise ValueError(
        "Final LBDINLC count differs from INS_J."
    )


print(
    "PASS: INS_J insulin variables retained."
)


# =================================================================================================
# 50. FINAL HbA1c CHECK
# =================================================================================================

if int(

    df["LBXGH"]
    .ne("")
    .sum()

) != 6045:

    raise ValueError(
        "Final LBXGH count differs from GHB_J."
    )


print(
    "PASS: LBXGH retained with 6,045 results."
)


# =================================================================================================
# 51. FINAL GLUCOSE CHECK
# =================================================================================================

if int(

    df["LBXGLU"]
    .ne("")
    .sum()

) != 2891:

    raise ValueError(
        "Final LBXGLU count differs."
    )


if int(

    df["LBDGLUSI"]
    .ne("")
    .sum()

) != 2891:

    raise ValueError(
        "Final LBDGLUSI count differs."
    )


print(
    "PASS: GLU_J measurements retained."
)


# =================================================================================================
# 52. FINAL CHOLESTEROL CHECK
# =================================================================================================

if int(

    df["LBXTC"]
    .ne("")
    .sum()

) != 6738:

    raise ValueError(
        "Final LBXTC count differs."
    )


if int(

    df["LBDHDD"]
    .ne("")
    .sum()

) != 6738:

    raise ValueError(
        "Final LBDHDD count differs."
    )


print(
    "PASS: Cholesterol measurements retained."
)


# =================================================================================================
# 53. FINAL TRIGLYCERIDE / LDL CHECK
# =================================================================================================

EXPECTED_FINAL_NONBLANK = {

    "LBXTR": 2834,

    "LBDTRSI": 2834,

    "LBDLDL": 2808,

    "LBDLDLSI": 2808,

    "LBDLDLM": 2808,

    "LBDLDMSI": 2808,

    "LBDLDLN": 2827,

    "LBDLDNSI": 2827

}


for col, expected_count in EXPECTED_FINAL_NONBLANK.items():

    actual_count = int(

        df[col]
        .ne("")
        .sum()

    )


    if actual_count != expected_count:

        raise ValueError(
            f"Final {col} count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: TRIGLY_J triglyceride and LDL variables retained."
)


# =================================================================================================
# 54. REPRESENTATIVE DECIMAL-PRESERVATION CHECK
# =================================================================================================

#
# Diagnostic only.
#
# Nothing is rounded or converted.
#

REPRESENTATIVE_DECIMAL_FIELDS = [

    "WTSAF2YR",

    "LBDGLUSI",

    "LBXIN",

    "LBDINSI",

    "LBXGH",

    "LBDHDDSI",

    "LBDTCSI",

    "LBDTRSI",

    "LBDLDLSI",

    "LBDLDMSI",

    "LBDLDNSI"

]


print(
    "\n"
    + "=" * 110
)


print(
    "LAB DECIMAL PRESERVATION"
)


print(
    "=" * 110
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in df.columns:

        continue


    populated_count = int(

        df[col]
        .ne("")
        .sum()

    )


    decimal_token_count = int(

        (
            df[col].ne("")
            &
            df[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: No Lab values were rounded or integerized "
    "during merge."
)


# =================================================================================================
# 55. KNOWN XPORT TINY-ARTIFACT CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_artifact_columns = {}


for col in df.columns:

    if col == "SEQN":

        continue


    count = int(

        df[col]

        .str.lower()

        .eq(
            TINY_TOKEN.lower()
        )

        .sum()

    )


    if count > 0:

        tiny_artifact_columns[
            col
        ] = count


tiny_artifact_count = sum(
    tiny_artifact_columns.values()
)


print(
    "\nKnown tiny XPORT artifacts:",
    f"{tiny_artifact_count:,}"
)


if tiny_artifact_count != 0:

    raise ValueError(
        "Known XPORT tiny-zero artifacts remain:\n"
        + str(
            tiny_artifact_columns
        )
    )


print(
    "PASS: No known XPORT tiny-zero artifacts."
)


# =================================================================================================
# 56. SAVE FINAL LAB CSV
# =================================================================================================

#
# NO float_format=
#
# NO rounding.
#

df.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved final Lab CSV:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 57. RE-READ FINAL CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 58. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != df.shape:

    raise ValueError(
        "CSV shape changed after save.\n"
        f"Before: {df.shape}\n"
        f"After:  {final_df.shape}"
    )


if (
    final_df.columns.tolist()
    !=
    df.columns.tolist()
):

    raise ValueError(
        "CSV column structure changed after save."
    )


print(
    "PASS: CSV structure preserved."
)


# =================================================================================================
# 59. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

post_save_failures = []


for col in df.columns:

    mismatch_count = int(

        (
            df[col]
            !=
            final_df[col]
        )

        .sum()

    )


    if mismatch_count > 0:

        post_save_failures.append(
            (
                col,
                mismatch_count
            )
        )


if post_save_failures:

    raise ValueError(
        "Text changed during CSV export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 60. FINAL SAVED SEQN CHECK
# =================================================================================================

if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Saved CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Saved CSV contains blank SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Saved CSV contains SEQN '.0'."
    )


if (

    ~final_df["SEQN"]
    .str.fullmatch(
        r"\d+"
    )

).any():

    raise ValueError(
        "Saved CSV contains invalid SEQN formatting."
    )


print(
    "PASS: Saved CSV SEQN integrity."
)


# =================================================================================================
# 61. FINAL DUPLICATE / OLD-CYCLE FIELD CHECK
# =================================================================================================

BAD_COLUMNS = [

    "WTSAF2YR_INS",

    "WTSAF2YR_TRIGLY",

    "WTSAF2YR_LBXGH",

    "WTSAF2YR_X",

    "WTSAF2YR_Y",

    "PHAFSTHR_FAST",

    "PHAFSTMN_FAST",

    "PHAFSTHR_GLU",

    "PHAFSTMN_GLU",

    "PHAFSTHR_INS",

    "PHAFSTMN_INS",

    "PHAFSTHR_X",

    "PHAFSTHR_Y",

    "PHAFSTMN_X",

    "PHAFSTMN_Y"

]


bad_columns_found = [

    col

    for col in BAD_COLUMNS

    if col in final_df.columns

]


if bad_columns_found:

    raise ValueError(
        "Unexpected duplicate/old-cycle fields remain:\n"
        + str(
            bad_columns_found
        )
    )


print(
    "PASS: No old-cycle or engineered duplicate fields."
)


# =================================================================================================
# 62. FINAL KEY FIELD CHECK
# =================================================================================================

EXPECTED_FINAL_FIELDS = [

    "WTSAF2YR",

    "LBXGLU",

    "LBDGLUSI",

    "LBXIN",

    "LBDINSI",

    "LBDINLC",

    "LBXGH",

    "PHAFSTHR",

    "PHAFSTMN",

    "LBXTR",

    "LBDTRSI",

    "LBDLDL",

    "LBDLDLSI",

    "LBDLDLM",

    "LBDLDMSI",

    "LBDLDLN",

    "LBDLDNSI",

    "LBXTC",

    "LBDTCSI",

    "LBDHDD",

    "LBDHDDSI"

]


for col in EXPECTED_FINAL_FIELDS:

    if col not in final_df.columns:

        raise ValueError(
            f"Expected final Lab variable missing: {col}"
        )


print(
    "PASS: All key 2017-2018 Lab variables present."
)


# =================================================================================================
# 63. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2017-2018 LAB MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    ""
)


print(
    "Source files:"
)


print(
    "  GLU_J      = 3,036"
)


print(
    "  INS_J      = 3,036"
)


print(
    "  TRIGLY_J   = 3,036"
)


print(
    "  TCHOL_J    = 7,435"
)


print(
    "  HDL_J      = 7,435"
)


print(
    "  GHB_J      = 6,401"
)


print(
    "  FASTQX_J   = 8,366"
)


print(
    ""
)


print(
    "WTSAF2YR:"
)


print(
    "  Canonical source = GLU_J"
)


print(
    "  INS_J copy validated then removed"
)


print(
    "  TRIGLY_J copy validated then removed"
)


print(
    "  Nonblank = 3,036"
)


print(
    "  Positive = 2,711"
)


print(
    "  Zero     = 325"
)


print(
    ""
)


print(
    "PHAFSTHR / PHAFSTMN:"
)


print(
    "  GLU_J versions = NOT PRESENT"
)


print(
    "  INS_J versions = NOT PRESENT"
)


print(
    "  Canonical source = FASTQX_J"
)


print(
    "  PHAFSTHR populated = 7,996"
)


print(
    "  PHAFSTMN populated = 7,996"
)


print(
    ""
)


print(
    "Glucose:"
)


print(
    "  LBXGLU populated   = 2,891"
)


print(
    "  LBDGLUSI populated = 2,891"
)


print(
    ""
)


print(
    "Insulin:"
)


print(
    "  LBXIN populated   = 2,825"
)


print(
    "  LBDINSI populated = 2,825"
)


print(
    "  LBDINLC populated = 2,825"
)


print(
    ""
)


print(
    "HbA1c:"
)


print(
    "  LBXGH populated = 6,045"
)


print(
    ""
)


print(
    "Lipids:"
)


print(
    "  LBXTC populated   = 6,738"
)


print(
    "  LBDHDD populated  = 6,738"
)


print(
    "  LBXTR populated   = 2,834"
)


print(
    "  LBDLDL populated  = 2,808"
)


print(
    "  LBDLDLM populated = 2,808"
)


print(
    "  LBDLDLN populated = 2,827"
)


print(
    ""
)


print(
    "WTSAF2YR GLU_J vs INS_J numeric mismatches:",
    wtsaf_ins_result[
        "numeric_mismatches"
    ]
)


print(
    "WTSAF2YR GLU_J vs TRIGLY_J numeric mismatches:",
    wtsaf_trigly_result[
        "numeric_mismatches"
    ]
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate column names: 0"
)


print(
    "Duplicate WTSAF2YR fields: 0"
)


print(
    "PHAFSTHR_FAST / PHAFSTMN_FAST fields: 0"
)


print(
    "Known tiny XPORT artifacts: 0"
)


print(
    "Lab decimal reformatting during merge: 0"
)


print(
    "Source-specific retained-field fidelity: PASS"
)


print(
    "Pre-save/post-save exact text fidelity: PASS"
)


print(
    "=" * 110
)

PASS: All seven 2017-2018 Lab source files exist.

NHANES 2017-2018 LAB MERGE


,file,rows,columns,column_names
0,GLU_J,3036,4,"[SEQN, WTSAF2YR, LBXGLU, LBDGLUSI]"
1,INS_J,3036,5,"[SEQN, WTSAF2YR, LBXIN, LBDINSI, LBDINLC]"
2,TCHOL_J,7435,3,"[SEQN, LBXTC, LBDTCSI]"
3,FASTQX_J,8366,19,"[SEQN, PHQ020, PHACOFHR, PHACOFMN, PHQ030, PHA..."
4,GHB_J,6401,2,"[SEQN, LBXGH]"
5,HDL_J,7435,3,"[SEQN, LBDHDD, LBDHDDSI]"
6,TRIGLY_J,3036,10,"[SEQN, WTSAF2YR, LBXTR, LBDTRSI, LBDLDL, LBDLD..."



--- OFFICIAL SOURCE ROW COUNTS ---
GLU_J: expected=3,036, actual=3,036
INS_J: expected=3,036, actual=3,036
TRIGLY_J: expected=3,036, actual=3,036
TCHOL_J: expected=7,435, actual=7,435
HDL_J: expected=7,435, actual=7,435
GHB_J: expected=6,401, actual=6,401
FASTQX_J: expected=8,366, actual=8,366
PASS: All source row counts match 2017-2018 expectations.

--- SEQN CHECK: GLU_J ---
Rows: 3,036
Unique SEQN: 3,036
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid format: 0
PASS

--- SEQN CHECK: INS_J ---
Rows: 3,036
Unique SEQN: 3,036
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid format: 0
PASS

--- SEQN CHECK: TCHOL_J ---
Rows: 7,435
Unique SEQN: 7,435
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid format: 0
PASS

--- SEQN CHECK: FASTQX_J ---
Rows: 8,366
Unique SEQN: 8,366
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid format: 0
PASS

--- SEQN CHECK: GHB_J ---
Rows: 6,401
Unique SEQN: 6,401
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0